# EXAMEN IA : Master 2, GROUPE 3
- RAZAFINIRINA Mahatradraibe Marcellin M000084 MF (45%)
- MAMINIRINA Hantamalala Stéphanie 6500 MF (55%)

In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
data = pd.read_csv("Data_2025.csv")

data

##  Extraction des données de Groupe3

In [ ]:
data3 = data.loc[16000:23999]
data3

In [ ]:
# changement de variable data3 par data
data = data3
data

In [ ]:
# enrengistrement des données
data.to_csv("groupe3.csv", index=False)

In [ ]:
# lire le data de groupe3
data = pd.read_csv("groupe3.csv")
data

In [ ]:
# vérification des valeurs manquantes
data.isnull().sum()

In [ ]:
data.info()

In [ ]:
# changement de categorie dans foret_fire par de valeurs quantitati ( nombre binaire ) pour faciliter le calcul et la classification
# Remplacer Y → 1 et N → 0
data["forest_fire"] = data["forest_fire"].map({"Y": 1, "N": 0})
data

In [ ]:
data.info()

## Affichage de la moyenne, variance, écart-type de chacune des variables quantitatives

In [ ]:
data.describe()

## Affichage des dimensions (nombre de lignes et de colonnes ) 

In [ ]:
data.shape
# Il y a 8000 lignes, 14 colonnes

## Choix des features et des targets

In [ ]:
# suppression des colonne date et time 
data1 = data.drop(["date","time"], axis=1)
# features et le target
X = data1.drop("forest_fire", axis=1) # features
y = data1["forest_fire"] # target

## Matrice de corrélation

In [ ]:
# Calcul de la matrice de corrélation
corr_matrix = data1.corr()

# Afficher la matrice (tableau)
print(corr_matrix)

In [ ]:
# Visualiser avec une heatmap
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f") # afficher avec 2 décimales
plt.title("Matrice de corrélation")
plt.show()

## Vusialisation de nombre d'incendie

In [ ]:
target_variable = 'forest_fire'
plt.figure()
sns.countplot(x=target_variable, data=data)
plt.title('Nombre d incendie')
plt.xlabel('incendie')
plt.ylabel('nombre')
plt.show()

## Prédiction des valeurs quantitatives

In [ ]:
# Séparation train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 20% de données pour faire le test.

In [ ]:
# Entrainement de modéle

# Initialiser le modèle
model = KNeighborsClassifier()

# Entraîner le modèle
model.fit(X_train, y_train)

In [ ]:
# prediction
y_pred = model.predict(X_test)


In [ ]:
# evaluation de modéle

# Exactitude globale
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy :", accuracy)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# Rapport de classification (precision, recall, f1-score)
report = classification_report(y_test, y_pred)
print("Classification Report:\n", report)

In [ ]:
# Optimisation
param_grid_model= {'n_neighbors':np.arange(1,50),
                  'weights':['uniform', 'distance'],
                  'p':[1,2]
                 }
grid_search_model = GridSearchCV(estimator=model, param_grid=param_grid_model, cv=5, scoring='accuracy')


In [ ]:
grid_search_model.fit(X_train, y_train)

In [ ]:
grid_search_model.best_score_

## Analyse des erreurs du modèle le plus performant sur l'ensemble de test 

In [ ]:
# Trouver les instances mal classées dans l'ensemble de test
misclassified_indices = np.where(y_test != y_pred)[0]
misclassified_samples = X_test.iloc[misclassified_indices]
misclassified_true_labels = y_test.iloc[misclassified_indices]
misclassified_predicted_labels = y_pred[misclassified_indices]

print("\nExemples d'instances mal classées (KNN) :")
misclassified_summary = pd.DataFrame({
    'Vraie Étiquette': misclassified_true_labels,
    'Étiquette Prédite': misclassified_predicted_labels
}, index=misclassified_samples.index)
display(misclassified_summary.head())


# Générer des courbes d'apprentissage pour le modèle le plus performant (Régression Logistique)
train_sizes, train_scores, test_scores = learning_curve(
    model, X_train, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy')

# Calculer la moyenne et l'écart type pour les scores d'entraînement et de test
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

# Tracer les courbes d'apprentissage
plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, alpha=0.1, color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r",
         label="Score d'entraînement")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g",
         label="Score de validation croisée")
plt.title("Courbe d'Apprentissage KNN")
plt.xlabel("Taille de l'Ensemble d'Entraînement")
plt.ylabel("Score d'exactitude")
plt.legend(loc="best")
plt.grid()
plt.show()